Cell 1 — Check environment

In [ ]:
import tensorflow as tf

print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

Cell 2 — Setup project root + import source code

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
    
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))



Cell 3 — Load dataset for CNN baseline

In [ ]:
from src.dataset import load_train_val_test_datasets
from src.train_cnn import train_cnn_model
from src.config import CNN_MODEL_PATH, CNN_HISTORY_FIGURE_PATH

train_ds, val_ds, test_ds, class_names = load_train_val_test_datasets()
num_classes = len(class_names)

print("Number of classes:", num_classes)
print("Model path:", CNN_MODEL_PATH)
print("History figure path:", CNN_HISTORY_FIGURE_PATH)

Cell 4 — Compute class weight

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

all_train_labels = []

for _, labels in train_ds:
    all_train_labels.extend(labels.numpy())

all_train_labels = np.array(all_train_labels)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(all_train_labels),
    y=all_train_labels
)

class_weight = {
    int(class_id): float(weight)
    for class_id, weight in zip(np.unique(all_train_labels), class_weights_array)
}

print("Number of class weights:", len(class_weight))

Cell 5 — Train CNN baseline

In [ ]:
train_ds_for_week2 = train_ds.take(700)
val_ds_for_week2 = val_ds.take(150)

EPOCHS = 20

cnn_model, cnn_history = train_cnn_model(
    train_ds=train_ds_for_week2,
    val_ds=val_ds_for_week2,
    num_classes=num_classes,
    epochs=EPOCHS,
    class_weight=class_weight
)

print("Model exists:", CNN_MODEL_PATH.exists())
print("Figure exists:", CNN_HISTORY_FIGURE_PATH.exists())

Cell 6 — Evaluate CNN baseline

In [ ]:
test_loss, test_accuracy = cnn_model.evaluate(test_ds)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)
print("Test accuracy percent:", test_accuracy * 100)

Cell 7 — Save CNN baseline report and confusion matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from src.config import FIGURES_DIR, REPORTS_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = cnn_model.predict(images, verbose=0)
    predicted_labels = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_labels)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
weighted_f1 = f1_score(y_true, y_pred, average="weighted")

print("Accuracy:", accuracy)
print("Macro-F1:", macro_f1)
print("Weighted-F1:", weighted_f1)

report = classification_report(y_true, y_pred, digits=4)

with open(REPORTS_DIR / "baseline_classification_report.txt", "w", encoding="utf-8") as f:
    f.write("CNN Baseline Classification Report\n")
    f.write("==================================\n\n")
    f.write(f"Accuracy: {accuracy:.4f}\n")
    f.write(f"Macro-F1: {macro_f1:.4f}\n")
    f.write(f"Weighted-F1: {weighted_f1:.4f}\n\n")
    f.write(report)

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax, values_format="d", xticks_rotation=90)
plt.title("CNN Baseline Confusion Matrix")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "baseline_confusion_matrix.png", dpi=300)
plt.show()